In [27]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import time

prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx"
historial = "detalle matricula sistemas pedacito.xlsx"
                                

In [28]:
class AnalizadorPrerrequisitos:
    def __init__(self):
        self.df_prereq = None
        self.df_historial = None
        self.resultado = None
        self.max_niveles_cadena = 5
        # Cache para optimización (no usado extensivamente aquí, pero lo dejo)
        self.cache_materias_estudiante = {}
        self.cache_prerrequisitos = {}
        # historial_optimizado será la tabla agregada por (Pidm, Cod materia curso)
        self.historial_optimizado = None
        # prereq_melted se crea en preparar_datos_optimizados
        self.prereq_melted = None

    def cargar_documentos(self):
        """Solicita y carga los documentos de Excel"""
        print("=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===\n")
        
        # Solicitar archivo de prerrequisitos
        while True:
            try:
                ruta_prereq = prereq #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
                self.df_prereq = pd.read_excel(ruta_prereq)
                print(f"✓ Archivo de prerrequisitos cargado: {len(self.df_prereq)} registros")
                break
            except Exception as e:
                print(f"Error al cargar prerrequisitos: {e}")
                print("Intente nuevamente.\n")
        
        # Solicitar archivo de historial
        while True:
            try:
                ruta_historial = historial#input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
                self.df_historial = pd.read_excel(ruta_historial)
                print(f"✓ Archivo de historial cargado: {len(self.df_historial)} registros")
                break
            except Exception as e:
                print(f"Error al cargar historial: {e}")
                print("Intente nuevamente.\n")

    def validar_columnas_y_datos(self):
        """Valida que existan las columnas necesarias y verifica duplicados"""
        print("Validando estructura de datos...")

        # Columnas requeridas en prerrequisitos
        cols_prereq_req = ['Smbarul_Key_Rule', 'Programa']
        cols_prereq_faltantes = [col for col in cols_prereq_req if col not in self.df_prereq.columns]
        if cols_prereq_faltantes:
            raise ValueError(f"Faltan columnas en prerrequisitos: {cols_prereq_faltantes}")

        # Columnas requeridas en historial
        cols_hist_req = ['Codigo_Programa', 'Cod materia curso', 'Pidm', 'Materia_Aprobada', 'Calificacion_Final']
        cols_hist_faltantes = [col for col in cols_hist_req if col not in self.df_historial.columns]

        # Verificar si existe columna Periodo
        if 'Periodo' not in self.df_historial.columns:
            print("⚠️  ADVERTENCIA: No se encontró la columna 'Periodo' en el historial.")
            print("Se procederá sin validación de duplicados por periodo. Se asumirá que 'Periodo' no aplica y se contará todo el historial como antes.")
            # No rompemos: se añadirá una columna Periodo ficticia en preparar_datos_optimizados para mantener la lógica
            self.validar_duplicados_sin_periodo()
        else:
            self.validar_duplicados_con_periodo()

        if cols_hist_faltantes:
            raise ValueError(f"Faltan columnas en historial: {cols_hist_faltantes}")

        print("✓ Validación completada")

    def validar_duplicados_con_periodo(self):
        """Valida duplicados usando Pidm + Periodo + Cod materia curso"""
        print("- Validando duplicados con columna Periodo...")

        combinacion_unica = ['Pidm', 'Periodo', 'Cod materia curso']
        duplicados = self.df_historial.duplicated(subset=combinacion_unica, keep=False)

        if duplicados.any():
            registros_duplicados = self.df_historial[duplicados]
            print(f"\n⚠️  ENCONTRADOS {duplicados.sum()} REGISTROS DUPLICADOS:")
            print("Registros con la misma combinación Pidm + Periodo + Cod materia curso:")

            for _, grupo in registros_duplicados.groupby(combinacion_unica):
                print(f"- Pidm: {grupo.iloc[0]['Pidm']}, Periodo: {grupo.iloc[0]['Periodo']}, "
                      f"Materia: {grupo.iloc[0]['Cod materia curso']} ({len(grupo)} registros)")

            print(f"\nEliminando {duplicados.sum()} registros duplicados...")
            self.df_historial = self.df_historial[~duplicados].copy()
            print(f"Registros restantes: {len(self.df_historial)}")

    def validar_duplicados_sin_periodo(self):
        """Información sobre duplicados cuando no existe la columna Periodo"""
        print("- Analizando repeticiones de Pidm + Cod materia curso...")

        combinaciones = self.df_historial.groupby(['Pidm', 'Cod materia curso']).size()
        repeticiones = combinaciones[combinaciones > 1]

        if len(repeticiones) > 0:
            print(f"ℹ️  Se encontraron {len(repeticiones)} combinaciones estudiante-materia con múltiples registros.")
            print("Esto es normal ya que los estudiantes pueden tomar la misma materia varias veces.")
            print(f"Total de registros extra por repeticiones: {repeticiones.sum() - len(repeticiones)}")
        else:
            print("ℹ️  No se encontraron repeticiones en Pidm + Cod materia curso.")

    def obtener_columnas_opciones(self):
        """Obtiene las columnas de opciones de prerrequisitos (Opcion_Prereq_1 a Opcion_Prereq_21)"""
        patron = r'^Opcion_Prereq_\d+$'
        columnas_opciones = [col for col in self.df_prereq.columns if re.match(patron, col)]
        columnas_opciones.sort(key=lambda x: int(x.split('_')[-1]))
        return columnas_opciones

    def parsear_prerrequisito(self, prereq_str):
        """Parsea una cadena de prerrequisito y devuelve una lista de códigos de asignaturas"""
        if pd.isna(prereq_str) or str(prereq_str).strip() == '':
            return []

        prereq_str = str(prereq_str).strip()

        if '&' in prereq_str:
            codigos = [codigo.strip() for codigo in prereq_str.split('&')]
            return [codigo for codigo in codigos if codigo]
        else:
            return [prereq_str] if prereq_str else []

    def preparar_datos_optimizados(self):
        """Prepara estructuras de datos optimizadas para el procesamiento.
           Además crea el contador de intentos acumulado (Intento) por (Pidm, Cod materia curso),
           y una tabla agregada (historial_optimizado) por (Pidm, Cod materia curso) con Aprobada/Nota_Final/Codigo_Programa/Intentos_total.
        """
        print("Preparando estructuras de datos optimizadas...")

        # Si no existe Periodo, crear uno ficticio muy grande para que las consultas por periodo incluyan todo el historial
        if 'Periodo' not in self.df_historial.columns:
            self.df_historial['Periodo'] = 999999
            print("⚠️  'Periodo' no estaba presente. Se ha creado una columna 'Periodo' ficticia para mantener compatibilidad.")

        # 0) Ordenar por Pidm, materia y periodo
        self.df_historial = self.df_historial.sort_values(by=['Pidm', 'Cod materia curso', 'Periodo'])

        # 1) Crear la columna 'Intento' (numero de intento acumulado por estudiante-materia en orden cronológico)
        self.df_historial['Intento'] = self.df_historial.groupby(['Pidm', 'Cod materia curso']).cumcount() + 1

        # 2) Crear tabla agregada por (Pidm, Cod materia curso) con Aprobada/Nota_Final/Codigo_Programa/Intentos_total
        def obtener_nota_final_grupo(g):
            aprobados = g[g['Materia_Aprobada'].astype(str).str.upper() == 'Y']
            if not aprobados.empty:
                return aprobados.iloc[0]['Calificacion_Final']
            else:
                return g.iloc[-1]['Calificacion_Final']

        def verificar_aprobacion_grupo(g):
            return (g['Materia_Aprobada'].astype(str).str.upper() == 'Y').any()

        agrup = self.df_historial.groupby(['Pidm', 'Cod materia curso'])
        historial_agg = pd.DataFrame({
            'Pidm': agrup.size().index.get_level_values(0),
            'Cod materia curso': agrup.size().index.get_level_values(1),
            'Aprobada': agrup['Materia_Aprobada'].apply(lambda s: (s.astype(str).str.upper() == 'Y').any()).values,
            'Nota_Final': agrup.apply(obtener_nota_final_grupo).values,
            'Codigo_Programa': agrup['Codigo_Programa'].first().values,
            'Intentos': agrup.size().values
        })

        # Reset index para obtener DataFrame bien formado
        historial_agg = historial_agg.reset_index(drop=True)

        # Guardar historial_optimizado (agregado) para merges que no necesiten periodo
        self.historial_optimizado = historial_agg

        # 3) Preparar tabla de prerrequisitos 'melted' (opciones)
        print("- Procesando opciones de prerrequisitos...")
        columnas_opciones = self.obtener_columnas_opciones()
        prereq_melted = []
        for _, row in self.df_prereq.iterrows():
            for i, col in enumerate(columnas_opciones, 1):
                if pd.notna(row[col]) and str(row[col]).strip():
                    prereq_melted.append({
                        'Smbarul_Key_Rule': row['Smbarul_Key_Rule'],
                        'Programa': row['Programa'],
                        'Opcion_Num': i,
                        'Prerrequisitos_Raw': str(row[col]).strip(),
                        'Prerrequisitos_Codigos': self.parsear_prerrequisito(row[col])
                    })
        self.prereq_melted = pd.DataFrame(prereq_melted)
        print(f"✓ Estructuras optimizadas creadas. Opciones de prerrequisitos: {len(self.prereq_melted)}")

    def verificar_opciones_prerrequisitos_vectorizado(self, estudiantes_materias):
        """
        Verifica todas las opciones de prerrequisitos respetando el 'Periodo' de la materia principal.
        estudiantes_materias debe contener columnas: ['Pidm', 'Cod materia curso', 'Periodo'].
        Retorna DataFrame con columnas:
        ['Pidm', 'Cod materia curso', 'Periodo', 'Estado', 'Opcion_Elegida',
         'Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerrequisitos_Intentos']
        """
        if estudiantes_materias.empty:
            return pd.DataFrame()

        # Asegurar que la columna Periodo exista en el input; si no, añadimos periodo alto (comportamiento anterior)
        if 'Periodo' not in estudiantes_materias.columns:
            estudiantes_materias = estudiantes_materias.copy()
            estudiantes_materias['Periodo'] = 999999

        print(f"- Verificando cumplimiento de opciones de prerrequisitos para {len(estudiantes_materias)} combinaciones (considerando Periodo)...")

        # Merge con opciones de prerrequisitos
        datos_con_prereq = estudiantes_materias.merge(
            self.prereq_melted,
            left_on='Cod materia curso',
            right_on='Smbarul_Key_Rule',
            how='left'
        )

        # Filtrar solo registros que tienen prerrequisitos
        datos_con_prereq = datos_con_prereq.dropna(subset=['Opcion_Num'])
        if datos_con_prereq.empty:
            return pd.DataFrame()

        # Expandir prerrequisitos por estudiante, opcion y periodo, y evaluar por periodo usando self.df_historial
        filas_expandidas = []
        for _, row in datos_con_prereq.iterrows():
            pidm = row['Pidm']
            materia_principal = row['Cod materia curso']  # materia principal
            opcion_num = row['Opcion_Num']
            periodo_main = row['Periodo']
            prerrequisitos = row['Prerrequisitos_Codigos'] or []

            if not isinstance(prerrequisitos, list) or len(prerrequisitos) == 0:
                continue

            for codigo_prereq in prerrequisitos:
                if codigo_prereq is None:
                    continue
                # Filtrar historial del prerrequisito hasta el periodo de la materia principal (<= periodo_main)
                hist_prereq = self.df_historial[
                    (self.df_historial['Pidm'] == pidm) &
                    (self.df_historial['Cod materia curso'] == codigo_prereq) &
                    (self.df_historial['Periodo'] <= periodo_main)
                ].sort_values(by=['Periodo', 'Intento'])
                encontrada = not hist_prereq.empty
                cumple = False
                nota_final_pr = None
                intentos_pr = None
                if encontrada:
                    # Si alguno de los intentos hasta ese periodo está aprobado -> cumple
                    aprobados = hist_prereq[hist_prereq['Materia_Aprobada'].astype(str).str.upper() == 'Y']
                    if not aprobados.empty:
                        cumple = True
                        nota_final_pr = aprobados.iloc[0]['Calificacion_Final']
                    else:
                        # no aprobado hasta periodo, nota final = ultima nota hasta periodo
                        nota_final_pr = hist_prereq.iloc[-1]['Calificacion_Final']
                    # intentos hasta ese periodo = max Intento
                    intentos_pr = int(hist_prereq['Intento'].max())
                filas_expandidas.append({
                    'Pidm': pidm,
                    'Materia_Principal': materia_principal,
                    'Periodo': periodo_main,
                    'Opcion_Num': opcion_num,
                    'Codigo_Prereq': codigo_prereq,
                    'Programa': row.get('Programa', None),
                    'Cumple': cumple,
                    'Encontrada': encontrada,
                    'Nota_Final': nota_final_pr,
                    'Intentos': intentos_pr
                })

        if not filas_expandidas:
            return pd.DataFrame()

        df_expandido = pd.DataFrame(filas_expandidas)

        # Agregar por estudiante, materia principal, opcion y periodo
        opciones_cumplimiento = df_expandido.groupby(['Pidm', 'Materia_Principal', 'Periodo', 'Opcion_Num']).agg({
            'Cumple': 'all',       # Todos los prerrequisitos de la opción deben cumplirse
            'Encontrada': 'all',   # Todos deben encontrarse (hasta el periodo)
            'Codigo_Prereq': lambda x: list(x),
            'Nota_Final': lambda x: list(x),
            'Intentos': lambda x: list(x)
        }).reset_index()

        # Para cada estudiante-materia-periodo, encontrar la primera opción que cumple
        resultado_final = []
        for (pidm, materia, periodo), grupo in opciones_cumplimiento.groupby(['Pidm', 'Materia_Principal', 'Periodo']):
            grupo_sorted = grupo.sort_values('Opcion_Num')
            opcion_elegida = None
            for _, opcion in grupo_sorted.iterrows():
                if opcion['Cumple'] and opcion['Encontrada']:
                    opcion_elegida = opcion
                    break
            if opcion_elegida is not None:
                resultado_final.append({
                    'Pidm': pidm,
                    'Cod materia curso': materia,
                    'Periodo': periodo,
                    'Opcion_Elegida': opcion_elegida['Opcion_Num'],
                    'Prerrequisitos_Codigos': opcion_elegida['Codigo_Prereq'],
                    'Prerrequisitos_Notas': opcion_elegida['Nota_Final'],
                    'Prerrequisitos_Intentos': opcion_elegida['Intentos'],
                    'Estado': 'Prerrequisito cumplido'
                })
            else:
                # No se encontró opción que cumpla -> devolvemos la primera opción (si existe)
                primera_opcion = grupo_sorted.iloc[0]
                resultado_final.append({
                    'Pidm': pidm,
                    'Cod materia curso': materia,
                    'Periodo': periodo,
                    'Opcion_Elegida': primera_opcion['Opcion_Num'],
                    'Prerrequisitos_Codigos': primera_opcion['Codigo_Prereq'],
                    'Prerrequisitos_Notas': primera_opcion['Nota_Final'],
                    'Prerrequisitos_Intentos': primera_opcion['Intentos'],
                    'Estado': 'Tiene prerrequisito y no se encontró en la historia'
                })

        return pd.DataFrame(resultado_final)

    def construir_cadenas_por_nivel(self, df_base, nivel=1, visited=None):
        """Construye las cadenas de prerrequisitos nivel por nivel (manteniendo Periodo)."""
        if visited is None:
            visited = set()

        if nivel > self.max_niveles_cadena or df_base is None or df_base.empty:
            return pd.DataFrame()

        print(f"- Procesando nivel {nivel} de cadena de prerrequisitos...")

        filas_nivel = []
        for _, row in df_base.iterrows():
            pidm = row['Pidm']
            # Materia_Original: si viene (de niveles previos) la usamos; si no, la materia principal es 'Cod materia curso'
            materia_original = row.get('Materia_Original', row.get('Cod materia curso'))
            periodo_main = row.get('Periodo', 999999)
            prerrequisitos_codigos = row.get('Prerrequisitos_Codigos', []) or []
            prerrequisitos_notas = row.get('Prerrequisitos_Notas', []) or []
            prerrequisitos_intentos = row.get('Prerrequisitos_Intentos', []) or []

            if not isinstance(prerrequisitos_codigos, list) or len(prerrequisitos_codigos) == 0:
                continue

            # Normalizar longitudes
            if not isinstance(prerrequisitos_notas, list):
                prerrequisitos_notas = [None] * len(prerrequisitos_codigos)
            if not isinstance(prerrequisitos_intentos, list):
                prerrequisitos_intentos = [None] * len(prerrequisitos_codigos)

            while len(prerrequisitos_notas) < len(prerrequisitos_codigos):
                prerrequisitos_notas.append(None)
            while len(prerrequisitos_intentos) < len(prerrequisitos_codigos):
                prerrequisitos_intentos.append(None)

            for i, codigo in enumerate(prerrequisitos_codigos):
                if codigo is None:
                    continue
                if isinstance(codigo, float) and np.isnan(codigo):
                    continue

                key = (pidm, materia_original, str(codigo), periodo_main)
                if key in visited:
                    continue
                visited.add(key)

                filas_nivel.append({
                    'Pidm': pidm,
                    'Materia_Original': materia_original,
                    'Cod materia curso': codigo,
                    'Nivel': nivel,
                    'Posicion': i + 1,
                    'Nota_Nivel': prerrequisitos_notas[i] if i < len(prerrequisitos_notas) else None,
                    'Intentos_Nivel': prerrequisitos_intentos[i] if i < len(prerrequisitos_intentos) else None,
                    'Estado_Nivel': 'Aprobada' if (i < len(prerrequisitos_notas) and pd.notna(prerrequisitos_notas[i])) else 'No Aprobada',
                    'Periodo': periodo_main
                })

        if not filas_nivel:
            return pd.DataFrame()

        df_nivel = pd.DataFrame(filas_nivel)

        # Buscar prerrequisitos del siguiente nivel: pasar (Pidm, Cod materia curso, Periodo)
        siguiente_nivel = self.verificar_opciones_prerrequisitos_vectorizado(
            df_nivel[['Pidm', 'Cod materia curso', 'Periodo']].drop_duplicates()
        )

        if siguiente_nivel.empty:
            return df_nivel

        # Asociar Materia_Original y Periodo para que los registros siguientes sepan a qué materia principal pertenecen
        mapa = df_nivel[['Pidm', 'Cod materia curso', 'Materia_Original', 'Periodo']].drop_duplicates(subset=['Pidm', 'Cod materia curso', 'Periodo'])
        siguiente_nivel = siguiente_nivel.merge(
            mapa,
            left_on=['Pidm', 'Cod materia curso', 'Periodo'],
            right_on=['Pidm', 'Cod materia curso', 'Periodo'],
            how='left'
        )

        # Recursión: procesar siguiente nivel y concatenar resultados (manteniendo visited)
        deeper = self.construir_cadenas_por_nivel(siguiente_nivel, nivel + 1, visited)

        if deeper is None or deeper.empty:
            return df_nivel

        return pd.concat([df_nivel, deeper], ignore_index=True)

    def procesar_prerrequisitos(self):
        """Procesa todos los registros de forma optimizada (respetando Periodo en los intentos)"""
        print("\n=== PROCESANDO PRERREQUISITOS CON INTENTOS POR PERIODO ===")
        inicio = time.time()

        self.validar_columnas_y_datos()
        self.preparar_datos_optimizados()

        # 1. Procesar prerrequisitos directos: ahora necesitamos Periodo para contar intentos hasta ese Periodo
        estudiantes_materias = self.df_historial[['Pidm', 'Cod materia curso', 'Periodo']].drop_duplicates()
        print(f"Procesando {len(estudiantes_materias)} combinaciones únicas estudiante-materia-periodo...")

        # Verificar prerrequisitos directos (respeta Periodo)
        prereq_directos = self.verificar_opciones_prerrequisitos_vectorizado(estudiantes_materias)

        # 2. Construir cadenas completas (solo si hay prerrequisitos directos)
        if not prereq_directos.empty:
            print("Construyendo cadenas completas de prerrequisitos...")
            cadenas_completas = self.construir_cadenas_por_nivel(prereq_directos.copy())
        else:
            print("No se encontraron prerrequisitos directos para procesar cadenas.")
            cadenas_completas = pd.DataFrame()

        # 3. Reconstruir resultado final
        print("Reconstruyendo datos finales...")
        self.resultado = self.df_historial.copy()

        # Agregar información de intentos de materia principal (hasta el periodo) -> usando columna Intento calculada en df_historial
        self.resultado = self.resultado.merge(
            self.df_historial[['Pidm', 'Cod materia curso', 'Periodo', 'Intento']].rename(columns={'Intento': 'Intentos_Materia_Principal'}),
            on=['Pidm', 'Cod materia curso', 'Periodo'],
            how='left'
        )

        # Agregar estados de prerrequisitos (unir por Pidm + Cod materia curso + Periodo)
        if not prereq_directos.empty:
            if not cadenas_completas.empty:
                # Agrupar cadenas por (Pidm, Materia_Original, Periodo) para obtener listas ordenadas por nivel y posicion
                agrupado = (
                    cadenas_completas
                    .sort_values(['Nivel', 'Posicion'])
                    .groupby(['Pidm', 'Materia_Original', 'Periodo'])
                    .agg({
                        'Cod materia curso': lambda x: list(x),
                        'Nota_Nivel': lambda x: list(x),
                        'Intentos_Nivel': lambda x: list(x),
                        'Nivel': lambda x: list(x)
                    })
                    .reset_index()
                )
                agrupado = agrupado.rename(columns={
                    'Materia_Original': 'Cod materia curso',
                    'Cod materia curso': 'Prerrequisitos_Codigos',
                    'Nota_Nivel': 'Prerrequisitos_Notas',
                    'Intentos_Nivel': 'Prerrequisitos_Intentos',
                    'Nivel': 'Prerrequisitos_Niveles'
                })
                agrupado['Prerrequisitos_Tipos'] = agrupado['Prerrequisitos_Niveles'].apply(
                    lambda L: ['Directo' if n == 1 else 'Cadena' for n in L]
                )
            else:
                # No cadenas: usamos la info de prereq_directos
                agrupado = prereq_directos[['Pidm', 'Cod materia curso', 'Periodo', 'Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerrequisitos_Intentos']].copy()
                agrupado = agrupado.rename(columns={'Cod materia curso': 'Cod materia curso'})
                agrupado['Prerrequisitos_Niveles'] = agrupado['Prerrequisitos_Codigos'].apply(
                    lambda x: [1] * len(x) if isinstance(x, list) else []
                )
                agrupado['Prerrequisitos_Tipos'] = agrupado['Prerrequisitos_Niveles'].apply(
                    lambda L: ['Directo' for _ in L]
                )

            # Añadir columna Estado/Opcion_Elegida de prereq_directos para la materia principal si existe (unir por Pidm, Cod materia curso, Periodo)
            estado_info = prereq_directos[['Pidm', 'Cod materia curso', 'Periodo', 'Estado', 'Opcion_Elegida']].copy()
            agrupado = agrupado.merge(estado_info, on=['Pidm', 'Cod materia curso', 'Periodo'], how='left')

            # Merge final en resultado (incluyendo Periodo)
            self.resultado = self.resultado.merge(
                agrupado,
                on=['Pidm', 'Cod materia curso', 'Periodo'],
                how='left'
            )

            # Expandir prerrequisitos a columnas separadas (se calculan intentos por periodo dentro)
            self.expandir_prerrequisitos_a_columnas()
        else:
            self.resultado['Estado'] = 'No tiene prerrequisito'

        # Llenar valores faltantes
        self.resultado['Estado'] = self.resultado['Estado'].fillna('No tiene prerrequisito')
        self.resultado['Intentos_Materia_Principal'] = self.resultado['Intentos_Materia_Principal'].fillna(1)

        tiempo_total = time.time() - inicio
        print(f"✓ Procesamiento completado en {tiempo_total:.2f} segundos")
        print(f"Velocidad: {len(self.resultado)/tiempo_total:.0f} registros/segundo")

    def expandir_prerrequisitos_a_columnas(self):
        """Expande las listas de prerrequisitos a columnas separadas (solo hasta lo necesario).
           Además agrega nivel y tipo por prerequisito y usa self.df_historial para calcular intentos hasta el periodo.
        """
        print("Expandiendo prerrequisitos a columnas separadas...")

        def safe_get_list_item(lista, index):
            if isinstance(lista, list) and len(lista) > index:
                return lista[index]
            return None

        if 'Prerrequisitos_Codigos' in self.resultado.columns:
            max_prereq = int(self.resultado['Prerrequisitos_Codigos'].dropna().apply(lambda x: len(x) if isinstance(x, list) else 0).max() or 0)
        else:
            max_prereq = 0

        for i in range(max_prereq):
            # Codigo y Nota provienen de las listas ya agrupadas (estas listas fueron construidas respetando Periodo)
            self.resultado[f'Prereq_{i+1}_Codigo'] = self.resultado['Prerrequisitos_Codigos'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Nota'] = self.resultado['Prerrequisitos_Notas'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Nivel'] = self.resultado['Prerrequisitos_Niveles'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_Tipo'] = self.resultado['Prerrequisitos_Tipos'].apply(lambda x: safe_get_list_item(x, i) if isinstance(x, list) else None)
            self.resultado[f'Prereq_{i+1}_EsCadena'] = self.resultado[f'Prereq_{i+1}_Tipo'].apply(lambda t: True if t == 'Cadena' else (False if pd.notna(t) else None))

            # Intentos del prerrequisito hasta el periodo de la materia principal (usando df_historial con la columna 'Intento')
            def get_intentos(row, idx=i):
                codigo = safe_get_list_item(row.get('Prerrequisitos_Codigos', []), idx)
                if codigo is None or pd.isna(codigo):
                    return None
                pidm = row['Pidm']
                periodo = row['Periodo'] if 'Periodo' in row else 999999
                intentos_df = self.df_historial[
                    (self.df_historial['Pidm'] == pidm) &
                    (self.df_historial['Cod materia curso'] == codigo) &
                    (self.df_historial['Periodo'] <= periodo)
                ]
                if intentos_df.empty:
                    return None
                return int(intentos_df['Intento'].max())

            self.resultado[f'Prereq_{i+1}_Intentos'] = self.resultado.apply(get_intentos, axis=1)

        # Contadores
        def count_prerequisites(lista):
            if isinstance(lista, list):
                return len(lista)
            return 0

        self.resultado['Num_Prerrequisitos_Total'] = self.resultado['Prerrequisitos_Codigos'].apply(count_prerequisites)

        def count_directs(niveles):
            if isinstance(niveles, list):
                return sum(1 for n in niveles if n == 1)
            return 0

        if 'Prerrequisitos_Niveles' in self.resultado.columns:
            self.resultado['Num_Prerrequisitos_Directos'] = self.resultado['Prerrequisitos_Niveles'].apply(lambda x: count_directs(x) if isinstance(x, list) else 0)
        else:
            self.resultado['Num_Prerrequisitos_Directos'] = 0

        # Limpiar columnas auxiliares
        columnas_a_eliminar = ['Prerrequisitos_Codigos', 'Prerrequisitos_Notas', 'Prerrequisitos_Intentos', 'Prerrequisitos_Niveles', 'Prerrequisitos_Tipos']
        for col in columnas_a_eliminar:
            if col in self.resultado.columns:
                self.resultado = self.resultado.drop(columns=[col])

    def generar_reporte(self):
        """Genera un reporte resumen del análisis"""
        if self.resultado is None:
            print("No hay resultados para reportar")
            return

        print("\n=== REPORTE DE RESULTADOS ===")

        resumen = self.resultado['Estado'].value_counts()
        print("\nResumen por estado:")
        for estado, cantidad in resumen.items():
            porcentaje = (cantidad / len(self.resultado)) * 100
            print(f"- {estado}: {cantidad} ({porcentaje:.1f}%)")

        if 'Num_Prerrequisitos_Directos' in self.resultado.columns:
            print(f"\nEstadísticas de prerrequisitos directos:")
            prereq_stats = self.resultado['Num_Prerrequisitos_Directos'].value_counts().sort_index()
            for num, cantidad in prereq_stats.items():
                print(f"- {num} prerrequisitos: {cantidad} materias")

        if 'Intentos_Materia_Principal' in self.resultado.columns:
            intentos_promedio = self.resultado['Intentos_Materia_Principal'].mean()
            print(f"\nPromedio de intentos por materia: {intentos_promedio:.2f}")

        print(f"\nTotal de registros analizados: {len(self.resultado)}")

    def guardar_resultados(self):
        """Guarda los resultados en un archivo Excel"""
        if self.resultado is None:
            print("No hay resultados para guardar")
            return

        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        nombre_archivo = "resultados_prerrequisitos_cadenas_"+timestamp+".xlsx"

        try:
            # Limpiar columnas completamente vacías
            self.resultado = self.resultado.dropna(axis=1, how='all')

            self.resultado.to_excel(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(self.resultado.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")

    def ejecutar(self):
        """Ejecuta el análisis completo optimizado"""
        try:
            self.cargar_documentos()
            self.procesar_prerrequisitos()
            self.generar_reporte()
            self.guardar_resultados()
            print("\n¡Análisis optimizado completado exitosamente!")
        except Exception as e:
            print(f"\nError durante la ejecución: {e}")
            import traceback
            traceback.print_exc()

# Función principal
def main():
    analizador = AnalizadorPrerrequisitos()
    analizador.ejecutar()

if __name__ == "__main__":
    main()


=== ANALIZADOR OPTIMIZADO DE PRERREQUISITOS CON CADENAS ===

✓ Archivo de prerrequisitos cargado: 3101 registros
✓ Archivo de historial cargado: 28512 registros

=== PROCESANDO PRERREQUISITOS CON INTENTOS POR PERIODO ===
Validando estructura de datos...
- Validando duplicados con columna Periodo...

⚠️  ENCONTRADOS 78 REGISTROS DUPLICADOS:
Registros con la misma combinación Pidm + Periodo + Cod materia curso:
- Pidm: 218222, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 234511, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 245010, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 248517, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 253992, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 286815, Periodo: 202330, Materia: IGL4920 (2 registros)
- Pidm: 293426, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 301015, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 324610, Periodo: 202510, Materia: IIN4319 (2 registros)
- Pidm: 334

C:\Users\vittorinor\AppData\Local\Temp\ipykernel_8548\3958521937.py:157: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'Nota_Final': agrup.apply(obtener_nota_final_grupo).values,


- Procesando opciones de prerrequisitos...
✓ Estructuras optimizadas creadas. Opciones de prerrequisitos: 2574
Procesando 28434 combinaciones únicas estudiante-materia-periodo...
- Verificando cumplimiento de opciones de prerrequisitos para 28434 combinaciones (considerando Periodo)...
Construyendo cadenas completas de prerrequisitos...
- Procesando nivel 1 de cadena de prerrequisitos...
- Verificando cumplimiento de opciones de prerrequisitos para 15729 combinaciones (considerando Periodo)...
- Procesando nivel 2 de cadena de prerrequisitos...
- Verificando cumplimiento de opciones de prerrequisitos para 9823 combinaciones (considerando Periodo)...
- Procesando nivel 3 de cadena de prerrequisitos...
- Verificando cumplimiento de opciones de prerrequisitos para 5054 combinaciones (considerando Periodo)...
- Procesando nivel 4 de cadena de prerrequisitos...
- Verificando cumplimiento de opciones de prerrequisitos para 2917 combinaciones (considerando Periodo)...
- Procesando nivel 5 de 